# Analisis de red social desde tweets recolectados

Objetivo: construir un grafo dirigido de interacciones (menciones, replies, retweets, quotes) para identificar actores clave que deben entrar en las listas de seguimiento.

In [10]:
import polars as pl
import networkx as nx
from pathlib import Path
import yaml

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

# -- Cargar todos los datasets de tweets --
parquet_files = [
    DATA_RAW / "seed_timelines_derecha.parquet",
    DATA_RAW / "seed_timelines_izquierda.parquet",
    DATA_RAW / "thanos_citizen_mentions.parquet",
    DATA_RAW / "thanos_cross_bloc.parquet",
    DATA_RAW / "thanos_quotes.parquet",
    DATA_RAW / "thanos_replies.parquet",
]

dfs = [pl.read_parquet(f) for f in parquet_files if f.exists()]
tweets = pl.concat(dfs, how="diagonal_relaxed").unique(subset=["tweet_id"])

# -- Eliminar usuario grok como autor y de menciones --
EXCLUDED_USERS = {"grok"}
tweets = (
    tweets
    .filter(~pl.col("author_username").str.to_lowercase().is_in(EXCLUDED_USERS))
    .with_columns(
        pl.col("mentions").list.eval(
            pl.element().filter(~pl.element().str.to_lowercase().is_in(EXCLUDED_USERS))
        ).alias("mentions")
    )
)

print(f"Tweets totales: {tweets.height:,}")
print(f"Autores unicos: {tweets['author_username'].n_unique():,}")
print(f"Columnas: {tweets.columns}")

Tweets totales: 9,126
Autores unicos: 4,692
Columnas: ['tweet_id', 'author_id', 'author_username', 'author_name', 'author_created_at', 'author_followers', 'author_following', 'author_tweet_count', 'author_verified', 'author_description', 'author_location', 'text', 'created_at', 'lang', 'source', 'conversation_id', 'in_reply_to_user_id', 'ref_type', 'ref_tweet_id', 'retweet_count', 'reply_count', 'like_count', 'quote_count', 'impression_count', 'hashtags', 'mentions', 'collected_at']


## Cargar bloques politicos desde candidates.yaml

In [11]:
# -- Mapeo handle -> bloque politico --
with open("../config/candidates.yaml") as f:
    cfg = yaml.safe_load(f)

bloque_map = {}  # username_lower -> bloque
for bloque_nombre, bloque_data in cfg["bloques"].items():
    for lista in [bloque_data.get("candidatos", []), bloque_data.get("cuentas_asociadas", [])]:
        for cuenta in lista:
            handle = cuenta["handle"].lstrip("@").lower()
            bloque_map[handle] = bloque_nombre

# Tabla de lookup id -> username desde los tweets
id_to_username = dict(
    tweets.select("author_id", "author_username")
    .unique(subset=["author_id"])
    .iter_rows()
)

print(f"Cuentas semilla mapeadas: {len(bloque_map)}")
print(f"IDs resueltos a username: {len(id_to_username)}")

Cuentas semilla mapeadas: 10
IDs resueltos a username: 4692


## Extraccion de aristas

Tres tipos de interaccion: mencion, reply, retweet/quote. Todas generan aristas dirigidas source -> target.

In [12]:
# -- 1. Menciones: author_username -> cada mencion en la lista --
edges_mentions = (
    tweets
    .filter(pl.col("mentions").list.len() > 0)
    .select("author_username", "mentions")
    .explode("mentions")
    .rename({"author_username": "source", "mentions": "target"})
    .with_columns(pl.lit("mencion").alias("type"))
)

# -- 2. Replies: author -> usuario al que responde --
# in_reply_to_user_id es un ID, resolver a username cuando sea posible
edges_replies = (
    tweets
    .filter(pl.col("ref_type") == "replied_to", pl.col("in_reply_to_user_id").is_not_null())
    .select("author_username", "in_reply_to_user_id")
    .with_columns(
        pl.col("in_reply_to_user_id")
        .replace(id_to_username)
        .alias("target")
    )
    .select(
        pl.col("author_username").alias("source"),
        "target",
        pl.lit("reply").alias("type"),
    )
)

# -- 3. Retweets y quotes: author -> autor original del tweet referenciado --
# Buscar el autor del tweet referenciado en nuestro dataset
tweet_id_to_author = dict(
    tweets.select("tweet_id", "author_username").iter_rows()
)

edges_rt_quote = (
    tweets
    .filter(pl.col("ref_type").is_in(["retweeted", "quoted"]), pl.col("ref_tweet_id").is_not_null())
    .select("author_username", "ref_tweet_id", "ref_type")
    .with_columns(
        pl.col("ref_tweet_id")
        .replace(tweet_id_to_author)
        .alias("target")
    )
    # Descartar las que no pudimos resolver (tweet original no esta en dataset)
    .filter(pl.col("target") != pl.col("ref_tweet_id"))
    .select(
        pl.col("author_username").alias("source"),
        "target",
        pl.col("ref_type").alias("type"),
    )
)

# -- Unir todas las aristas --
all_edges = pl.concat([edges_mentions, edges_replies, edges_rt_quote])

# Normalizar a lowercase para evitar duplicados por capitalizacion
all_edges = all_edges.with_columns(
    pl.col("source").str.to_lowercase(),
    pl.col("target").str.to_lowercase(),
)

# Eliminar self-loops
all_edges = all_edges.filter(pl.col("source") != pl.col("target"))

print(f"Aristas totales: {all_edges.height:,}")
print(f"Por tipo:")
print(all_edges["type"].value_counts().sort("count", descending=True))

Aristas totales: 20,003
Por tipo:
shape: (4, 2)
┌───────────┬───────┐
│ type      ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ mencion   ┆ 16919 │
│ reply     ┆ 2204  │
│ retweeted ┆ 451   │
│ quoted    ┆ 429   │
└───────────┴───────┘


## Construccion del grafo dirigido ponderado

In [13]:
# Agregar aristas: peso = numero de interacciones entre par (source, target)
weighted_edges = (
    all_edges
    .group_by("source", "target")
    .agg(
        pl.len().alias("weight"),
        pl.col("type").value_counts().alias("type_counts"),
    )
)

G = nx.DiGraph()

# Agregar aristas con peso
for row in weighted_edges.iter_rows(named=True):
    G.add_edge(row["source"], row["target"], weight=row["weight"])

# Atributos de nodo: bloque politico y si es cuenta semilla
author_meta = (
    tweets
    .select("author_username", "author_followers", "author_following")
    .with_columns(pl.col("author_username").str.to_lowercase())
    .group_by("author_username")
    .agg(
        pl.col("author_followers").max(),
        pl.col("author_following").max(),
    )
)
followers_map = dict(zip(
    author_meta["author_username"].to_list(),
    author_meta["author_followers"].to_list(),
))

for node in G.nodes():
    G.nodes[node]["bloque"] = bloque_map.get(node, "desconocido")
    G.nodes[node]["es_semilla"] = node in bloque_map
    G.nodes[node]["followers"] = followers_map.get(node, 0)

print(f"Nodos: {G.number_of_nodes():,}")
print(f"Aristas: {G.number_of_edges():,}")
print(f"Nodos semilla: {sum(1 for _, d in G.nodes(data=True) if d['es_semilla'])}")
print(f"Nodos ciudadanos: {sum(1 for _, d in G.nodes(data=True) if not d['es_semilla'])}")

Nodos: 5,937
Aristas: 12,402
Nodos semilla: 10
Nodos ciudadanos: 5927


## Metricas de centralidad

In [14]:
in_degree = dict(G.in_degree(weight="weight"))
pagerank = nx.pagerank(G, weight="weight")
betweenness = nx.betweenness_centrality(G, weight="weight")

metrics = pl.DataFrame({
    "username": list(G.nodes()),
    "in_degree": [in_degree[n] for n in G.nodes()],
    "pagerank": [pagerank[n] for n in G.nodes()],
    "betweenness": [betweenness[n] for n in G.nodes()],
    "bloque": [G.nodes[n]["bloque"] for n in G.nodes()],
    "es_semilla": [G.nodes[n]["es_semilla"] for n in G.nodes()],
    "followers": [G.nodes[n]["followers"] for n in G.nodes()],
})

print("-- Top 20 por in-degree (quien recibe mas interacciones) --")
print(metrics.sort("in_degree", descending=True).head(20))

print("\n-- Top 20 por PageRank --")
print(metrics.sort("pagerank", descending=True).head(20))

print("\n-- Top 20 por betweenness (puentes entre comunidades) --")
print(metrics.sort("betweenness", descending=True).head(20))

-- Top 20 por in-degree (quien recibe mas interacciones) --
shape: (20, 7)
┌─────────────────┬───────────┬──────────┬─────────────┬─────────────┬────────────┬───────────┐
│ username        ┆ in_degree ┆ pagerank ┆ betweenness ┆ bloque      ┆ es_semilla ┆ followers │
│ ---             ┆ ---       ┆ ---      ┆ ---         ┆ ---         ┆ ---        ┆ ---       │
│ str             ┆ i64       ┆ f64      ┆ f64         ┆ str         ┆ bool       ┆ i64       │
╞═════════════════╪═══════════╪══════════╪═════════════╪═════════════╪════════════╪═══════════╡
│ ivancepedacast  ┆ 2615      ┆ 0.081392 ┆ 0.025404    ┆ izquierda   ┆ true       ┆ 1933160   │
│ abdelaespriella ┆ 1711      ┆ 0.045172 ┆ 0.027077    ┆ derecha     ┆ true       ┆ 166543    │
│ palomavalencial ┆ 1602      ┆ 0.05044  ┆ 0.016665    ┆ derecha     ┆ true       ┆ 641071    │
│ jdoviedoar      ┆ 706       ┆ 0.033202 ┆ 0.021352    ┆ derecha     ┆ true       ┆ 167168    │
│ petrogustavo    ┆ 690       ┆ 0.020171 ┆ 0.008317    ┆ izqu

## Deteccion de comunidades (Louvain)

In [15]:
from networkx.algorithms.community import louvain_communities

# Louvain opera sobre grafo no dirigido
G_undirected = G.to_undirected()
communities = louvain_communities(G_undirected, weight="weight", seed=42)

# Asignar comunidad a cada nodo
community_map = {}
for i, comm in enumerate(communities):
    for node in comm:
        community_map[node] = i

for node in G.nodes():
    G.nodes[node]["comunidad"] = community_map.get(node, -1)

print(f"Comunidades detectadas: {len(communities)}")
for i, comm in enumerate(sorted(communities, key=len, reverse=True)[:10]):
    semillas = [n for n in comm if n in bloque_map]
    print(f"  Comunidad {i}: {len(comm)} nodos, semillas: {semillas[:5]}")

Comunidades detectadas: 80
  Comunidad 0: 891 nodos, semillas: ['pizarromariajo', 'ivancepedacast']
  Comunidad 1: 713 nodos, semillas: ['cedemocratico', 'alvarouribevel', 'palomavalencial']
  Comunidad 2: 626 nodos, semillas: ['abdelaespriella']
  Comunidad 3: 461 nodos, semillas: ['gustavobolivar']
  Comunidad 4: 457 nodos, semillas: ['jdoviedoar']
  Comunidad 5: 392 nodos, semillas: ['pactohistorico']
  Comunidad 6: 391 nodos, semillas: []
  Comunidad 7: 320 nodos, semillas: ['petrogustavo']
  Comunidad 8: 235 nodos, semillas: []
  Comunidad 9: 228 nodos, semillas: []


## Afinidad politica por proximidad en la red

Para cada nodo se calcula la proporcion de interacciones (in + out) con cuentas de derecha vs izquierda.
Score continuo en [-1, +1]: -1 = pura izquierda, +1 = pura derecha, 0 = neutro.

In [16]:
# Construir set de nodos por bloque para lookup rapido
nodos_derecha = {n for n, b in bloque_map.items() if b == "derecha"}
nodos_izquierda = {n for n, b in bloque_map.items() if b == "izquierda"}

afinidad = {}
for node in G.nodes():
    # Semillas conservan su bloque directamente
    if node in bloque_map:
        score = 1.0 if bloque_map[node] == "derecha" else -1.0
        afinidad[node] = score
        continue

    peso_der = 0
    peso_izq = 0
    # Aristas salientes
    for _, target, data in G.out_edges(node, data=True):
        w = data.get("weight", 1)
        if target in nodos_derecha:
            peso_der += w
        elif target in nodos_izquierda:
            peso_izq += w
    # Aristas entrantes
    for source, _, data in G.in_edges(node, data=True):
        w = data.get("weight", 1)
        if source in nodos_derecha:
            peso_der += w
        elif source in nodos_izquierda:
            peso_izq += w

    total = peso_der + peso_izq
    if total > 0:
        afinidad[node] = (peso_der - peso_izq) / total
    else:
        afinidad[node] = 0.0

# Clasificacion discreta
def clasificar(score):
    if score > 0.3:
        return "derecha"
    elif score < -0.3:
        return "izquierda"
    return "neutro"

# Asignar al grafo
for node in G.nodes():
    G.nodes[node]["afinidad_score"] = round(afinidad[node], 4)
    G.nodes[node]["afinidad_label"] = clasificar(afinidad[node])
    G.nodes[node]["label"] = node  # username visible en Gephi

# Resumen
from collections import Counter
conteo = Counter(G.nodes[n]["afinidad_label"] for n in G.nodes())
print(f"Distribucion de afinidad:")
for label, count in conteo.most_common():
    print(f"  {label}: {count}")

Distribucion de afinidad:
  neutro: 3100
  derecha: 1440
  izquierda: 1397


## Actores clave no-semilla (candidatos para listas de seguimiento)

In [17]:
# Actores que NO son cuentas semilla pero tienen alta centralidad
actores_clave = (
    metrics
    .filter(~pl.col("es_semilla"))
    .with_columns(
        # Score compuesto: normalizar cada metrica y promediar
        ((pl.col("in_degree") / pl.col("in_degree").max())
         + (pl.col("pagerank") / pl.col("pagerank").max())
         + (pl.col("betweenness") / pl.col("betweenness").max()))
        .alias("score_compuesto")
    )
    # Agregar afinidad desde el grafo
    .with_columns(
        pl.col("username").replace({n: G.nodes[n]["afinidad_label"] for n in G.nodes()}).alias("afinidad"),
        pl.col("username").replace({n: G.nodes[n]["afinidad_score"] for n in G.nodes()}).alias("afinidad_score"),
    )
    .sort("score_compuesto", descending=True)
)

print("Top 30 actores no-semilla por score compuesto:")
print(actores_clave.head(30).select("username", "in_degree", "pagerank", "betweenness", "score_compuesto", "afinidad", "followers"))

Top 30 actores no-semilla por score compuesto:
shape: (30, 7)
┌─────────────────┬───────────┬──────────┬─────────────┬─────────────────┬───────────┬───────────┐
│ username        ┆ in_degree ┆ pagerank ┆ betweenness ┆ score_compuesto ┆ afinidad  ┆ followers │
│ ---             ┆ ---       ┆ ---      ┆ ---         ┆ ---             ┆ ---       ┆ ---       │
│ str             ┆ i64       ┆ f64      ┆ f64         ┆ f64             ┆ str       ┆ i64       │
╞═════════════════╪═══════════╪══════════╪═════════════╪═════════════════╪═══════════╪═══════════╡
│ aida_quilcue    ┆ 196       ┆ 0.012595 ┆ 0.0         ┆ 2.0             ┆ izquierda ┆ 0         │
│ izquierdosky_40 ┆ 104       ┆ 0.005593 ┆ 0.006994    ┆ 1.974702        ┆ izquierda ┆ 14541     │
│ camiloromero    ┆ 187       ┆ 0.00641  ┆ 0.003302    ┆ 1.935211        ┆ izquierda ┆ 337938    │
│ jrestrp         ┆ 175       ┆ 0.008379 ┆ 0.000019    ┆ 1.560847        ┆ derecha   ┆ 127121    │
│ platom___       ┆ 134       ┆ 0.005984 ┆ 0.00

In [33]:
actores_clave.head()

username,in_degree,pagerank,betweenness,bloque,es_semilla,followers,score_compuesto,afinidad,afinidad_score
str,i64,f64,f64,str,bool,i64,f64,str,f32
"""aida_quilcue""",196,0.012595,0.0,"""desconocido""",false,0,2.0,"""izquierda""",-0.8148
"""izquierdosky_40""",104,0.005593,0.006994,"""desconocido""",false,14541,1.974702,"""izquierda""",-0.4286
"""camiloromero""",187,0.00641,0.003302,"""desconocido""",false,337938,1.935211,"""izquierda""",-1.0
"""jrestrp""",175,0.008379,0.000019,"""desconocido""",false,127121,1.560847,"""derecha""",1.0
"""platom___""",134,0.005984,0.002185,"""desconocido""",false,48809,1.471201,"""derecha""",1.0


In [32]:
actores_clave = actores_clave.cast({"afinidad_score": pl.Float32})

In [40]:

actores_clave.filter( (pl.col("followers")>50000)
                      )

username,in_degree,pagerank,betweenness,bloque,es_semilla,followers,score_compuesto,afinidad,afinidad_score
str,i64,f64,f64,str,bool,i64,f64,str,f32
"""camiloromero""",187,0.00641,0.003302,"""desconocido""",false,337938,1.935211,"""izquierda""",-1.0
"""jrestrp""",175,0.008379,0.000019,"""desconocido""",false,127121,1.560847,"""derecha""",1.0
"""emiliolagos1""",185,0.004189,0.001259,"""desconocido""",false,76997,1.456438,"""izquierda""",-1.0
"""matador000""",129,0.004585,0.002035,"""desconocido""",false,501975,1.313149,"""neutro""",0.0
"""wilsonariasc""",71,0.003514,0.002529,"""desconocido""",false,377194,1.002933,"""izquierda""",-0.3333
…,…,…,…,…,…,…,…,…,…
"""johaypunto""",0,0.000063,0.0,"""desconocido""",false,55574,0.005021,"""neutro""",0.0
"""tercer_canal""",0,0.000063,0.0,"""desconocido""",false,76886,0.005021,"""derecha""",1.0
"""radioguatapuri""",0,0.000063,0.0,"""desconocido""",false,88728,0.005021,"""derecha""",1.0


## Exportacion GEXF para Gephi

Atributos de nodo exportados: bloque, es_semilla, followers, comunidad, pagerank, in_degree.

In [18]:
# Agregar metricas como atributos de nodo para Gephi
for node in G.nodes():
    G.nodes[node]["pagerank"] = pagerank.get(node, 0)
    G.nodes[node]["in_degree_w"] = in_degree.get(node, 0)
    G.nodes[node]["betweenness"] = betweenness.get(node, 0)

output_path = DATA_PROCESSED / "red_menciones.gexf"
nx.write_gexf(G, output_path)
print(f"Grafo exportado a: {output_path}")
print(f"  Nodos: {G.number_of_nodes():,}")
print(f"  Aristas: {G.number_of_edges():,}")

# Tambien guardar aristas y metricas como parquet
all_edges.write_parquet(DATA_PROCESSED / "mention_edges.parquet")
metrics.write_parquet(DATA_PROCESSED / "mention_metrics.parquet")
actores_clave.head(100).write_parquet(DATA_PROCESSED / "actores_clave_listas.parquet")
print("Parquets guardados: mention_edges, mention_metrics, actores_clave_listas")

Grafo exportado a: ../data/processed/red_menciones.gexf
  Nodos: 5,937
  Aristas: 12,402
Parquets guardados: mention_edges, mention_metrics, actores_clave_listas
